In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

y_train_onehot = keras.utils.to_categorical(y_train, num_classes=10)
y_test_onehot = keras.utils.to_categorical(y_test, num_classes=10)

In [2]:
model_base = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_base.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])
model_base.fit(x_train, y_train_onehot, epochs=5, batch_size=128, verbose=0)

y_pred_probs = model_base.predict(x_test)

Y_pred = np.argmax(y_pred_probs, axis=1)
Y_actual = y_test 

conf_matrix = pd.crosstab(Y_actual, Y_pred, rownames=['actual'], colnames=['pred'])
print(conf_matrix)

C:\Users\fabio\anaconda3\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
pred      0     1    2    3    4    5    6    7    8    9
actual                                                   
0       958     1    1    1    1    2    4    4    5    3
1         0  1115    0    4    1    2    1    2   10    0
2         4     4  954   23    2    1    4   13   25    2
3         0     0   13  963    0    4    0    7   19    4
4         2     0    6    1  885    0    6    5   18   59
5         1     1    2   55    0  810   10    0   10    3
6        22     3    6    2    3   10  897    0   14    1
7         1     7   12   10    0    1    0  981    6   10
8         5     2    9   22    2   12    4    6  904    8
9         4     4    0    9   11    5    0   20   11  945


In [3]:
# 01.3.2: Variante mit Keras Rescaling-Layer 
model_norm = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Rescaling(1./255),  # skaliert Pixelwerte automatisch auf [0, 1]
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_norm.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_norm = model_norm.fit(
    x_train, 
    y_train_onehot, 
    epochs=10, 
    batch_size=128, 
    validation_data=(x_test, y_test_onehot)
)

loss_norm, acc_norm = model_norm.evaluate(x_test, y_test_onehot)
print(f"Testgenauigkeit (Normalisiert): {acc_norm:.4f}")

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8948 - loss: 0.3905 - val_accuracy: 0.9327 - val_loss: 0.2292
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9428 - loss: 0.2020 - val_accuracy: 0.9483 - val_loss: 0.1728
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9548 - loss: 0.1556 - val_accuracy: 0.9581 - val_loss: 0.1449
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9624 - loss: 0.1284 - val_accuracy: 0.9623 - val_loss: 0.1298
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9678 - loss: 0.1096 - val_accuracy: 0.9673 - val_loss: 0.1115
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9722 - loss: 0.0963 - val_accuracy: 0.9690 - val_loss: 0.1073
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9755 - loss: 0.0853 - val_accuracy: 0.9691 - val_loss: 0.1018
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9779 - loss: 0.0767 - val_accuracy: 0.

In [4]:
# 01.3.3: Höhere Netzkomplexität (mehr Neuronen + zusätzlicher Layer + Dropout) & EarlyStopping
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True
)

model_opt = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Rescaling(1./255),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),  # Hilft zusätzlich gegen Overfitting
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_opt.compile(
    optimizer="adam",  # Adam funktioniert bei tieferen Netzen oft noch besser als rmsprop
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_opt = model_opt.fit(
    x_train, 
    y_train_onehot, 
    epochs=50, 
    batch_size=128, 
    validation_data=(x_test, y_test_onehot),
    callbacks=[early_stop]
)

test_loss, test_acc = model_opt.evaluate(x_test, y_test_onehot)
print(f"Maximierte Testgenauigkeit: {test_acc:.4f} (Loss: {test_loss:.4f})")

Epoch 1/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9111 - loss: 0.3028 - val_accuracy: 0.9617 - val_loss: 0.1309
Epoch 2/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9618 - loss: 0.1251 - val_accuracy: 0.9717 - val_loss: 0.0902
Epoch 3/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9726 - loss: 0.0893 - val_accuracy: 0.9717 - val_loss: 0.0883
Epoch 4/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9776 - loss: 0.0699 - val_accuracy: 0.9762 - val_loss: 0.0727
Epoch 5/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9814 - loss: 0.0576 - val_accuracy: 0.9759 - val_loss: 0.0770
Epoch 6/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9847 - loss: 0.0468 - val_accuracy: 0.9810 - val_loss: 0.0623
Epoch 7/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9865 - loss: 0.0411 - val_accuracy: 0.9803 - val_loss: 0.0656
Epoch 8/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9876 - loss: 0.0373 - val_accuracy: 0.

In [5]:
# 01.3.4: Modell im aktuellen Keras-Format speichern
model_opt.save("mnist_modell.keras")
print("Modell gespeichert.")

# Aktuelle Sequential-Instanz explizit löschen
del model_opt

# Modell wieder aus der Datei laden
loaded_model = keras.models.load_model("mnist_modell.keras")
print("Modell erfolgreich neu geladen.")

# Gegenprüfung, ob das geladene Modell exakt dieselbe Genauigkeit liefert
loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test_onehot)
print(f"Testgenauigkeit geladenes Modell: {loaded_acc:.4f}")

Modell gespeichert.
Modell erfolgreich neu geladen.
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9833 - loss: 0.0644
Testgenauigkeit geladenes Modell: 0.9833


### Ergebnisse 01.3.3

* **Normalisierung (`Rescaling(1./255)`):** Mit `Dense(64, relu)` + `Dense(10, softmax)` und `rmsprop` stieg die Testgenauigkeit über 10 Epochen auf **0.9726**.
* **Optimiertes Modell:** Mit `Dense(256, relu)`, `Dropout(0.2)`, `Dense(128, relu)`, `adam` und `EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)` stoppte das Training bei Epoche 18/50 und erreichte eine Testgenauigkeit von **0.9833** (Loss: **0.0644**).
* **Fazit:** Skalierung auf `[0, 1]`, mehr Kapazität mit Dropout gegen Overfitting, Adam und EarlyStopping haben die Genauigkeit deutlich und ohne starkes Overfitting verbessert.